# Lesson 06 · Backpropagation: writing the backward pass of the MNIST network yourself

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/trungnguyenvn/hoc-sau-notebooks/blob/main/06-lan-truyen-nguoc-en.ipynb)

This notebook accompanies [Lesson 06 of Học Sâu · Reels](https://d3qr1cjln6qqtk.cloudfront.net/en/lessons/lan-truyen-nguoc/), a deep-learning course originally in
Vietnamese. It is the English edition of the course's notebook
([Vietnamese original](https://github.com/trungnguyenvn/hoc-sau-notebooks/blob/main/06-lan-truyen-nguoc.ipynb)):
the code, the data and every check are the same. The reel and the interactive demo are on the lesson page.
The course's 784 → 128 → 64 → 10 MNIST network has more than a hundred thousand parameters. After each batch of
images, we need to know, for *each* parameter: if we increase it a little, does the loss go up or down, and how fast?
We write the algorithm that answers this question for every parameter at once ourselves, in numpy, check it number by
number, then use it to train the network on real MNIST data.

**What you'll do:**
1. Run the forward pass and the backward pass by hand on the demo's 7-parameter graph, then check each gradient by
   nudging.
2. Write a tiny autograd (micrograd-style) to see how the chain rule gets automated.
3. Write the backward pass in matrix form for the 784 → 128 → 64 → 10 network and check all 6 parameter tensors with
   finite differences.
4. Count the cost: the multiply-accumulate (MAC) operations of the backward pass against those of the forward pass,
   and the memory needed to keep the intermediate values.
5. Train the network for 5 epochs with your own backward pass, and get exactly the numbers of the reel and the demo.

**Background:** [Lesson 04 · Softmax & loss functions](https://d3qr1cjln6qqtk.cloudfront.net/en/lessons/ham-mat-mat/)
(cross-entropy) and [Lesson 05 · Gradient descent](https://d3qr1cjln6qqtk.cloudfront.net/en/lessons/gradient-descent/)
(the update step $\theta \leftarrow \theta - \eta\,\nabla_\theta \mathcal{L}$). ReLU comes from
[Lesson 02](https://d3qr1cjln6qqtk.cloudfront.net/en/lessons/ham-kich-hoat/), multilayer networks from
[Lesson 03](https://d3qr1cjln6qqtk.cloudfront.net/en/lessons/mlp-va-xor/).
**Run time:** under a minute on a CPU, including the first MNIST download (~11 MB).

The notation follows the *Neural Computation* chapter of [MLSysBook](https://mlsysbook.ai/vol1/nn_computation/nn_computation.html),
with the row-vector convention $z = xW + b$, as in every previous lesson.
Every number in the notebook is computed by the cells below, and matches the numbers in the reel and the demo.

> The **🤔 Predict first** cells ask a question before you run something. Answer it yourself, then run the next cell.

## 1 · Setup

Only numpy and matplotlib are needed (Colab has them preinstalled). The function `num()` prints numbers in the
English format: a decimal point, and commas between thousands. `sci()` prints very small numbers in the form
$3.5 \cdot 10^{-6}$. That is the same point Python uses when you *type* a decimal (`0.1`); printing only adds the
thousands separators and a real minus sign.

In [ ]:
import gzip
import hashlib
import itertools
import math
import textwrap
import time
import urllib.request
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
from matplotlib.lines import Line2D
from matplotlib.patches import FancyBboxPatch
from matplotlib.ticker import FuncFormatter, ScalarFormatter

# The course's palette (colorblind-safe), always paired with a second encoding (marker shape, dash style, label)
BLUE, ORANGE, GREEN, GREY = "#3987e5", "#d95926", "#199e70", "#8a94a6"
plt.rcParams.update({"figure.dpi": 110, "axes.spines.top": False, "axes.spines.right": False})


def num(x, digits=2):
    """A number in the English format: 13,007 · 0.1 · −4.38."""
    return f"{x:,.{digits}f}".replace("-", "−")


def pct(p, digits=2):
    return num(100 * p, digits) + "%"


def tick(v, _pos=None):
    """An axis tick label: 0.5 · −2 · 60,000 (no trailing zeros)."""
    if abs(v) < 1e-12:
        return "0"
    s = f"{v:.4f}".rstrip("0").rstrip(".")
    return num(v, len(s.split(".")[1]) if "." in s else 0)


def fmt_axes(fig):
    """Put the tick labels of every linear axis (colorbars included) in the English format; call right before
    plt.show()."""
    for ax in fig.axes:
        for axis in (ax.xaxis, ax.yaxis):
            # only axes still on matplotlib's default number formatter: text labels and custom formatters stay
            if axis.get_scale() == "linear" and isinstance(axis.get_major_formatter(), ScalarFormatter):
                axis.set_major_formatter(FuncFormatter(tick))


SUP = str.maketrans("-0123456789", "⁻⁰¹²³⁴⁵⁶⁷⁸⁹")


def sci(x, digits=1):
    """A very small or very large number: 3.5·10⁻⁶."""
    if x == 0:
        return "0"
    m, e = f"{x:.{digits}e}".split("e")
    return f"{num(float(m), digits)}·10{str(int(e)).translate(SUP)}"


print(num(109386, 0), num(0.1, 1), num(-0.5399, 3), pct(0.9727), sci(3.5e-6))

## 2 · The MNIST data and the lesson's question

MNIST (LeCun et al., 1998) has 70,000 grayscale 28×28 images of handwritten digits: 60,000 training images and
10,000 test images. The cell below downloads the 4 original files from a public mirror and checks their MD5 hashes,
so everyone who runs it gets exactly the same data. Each image is laid out as a row of 784 numbers in $[0, 1]$, as in
Lesson 01.

In [ ]:
# MNIST: exactly as in reel/perceptron/data.py
URL = "https://ossci-datasets.s3.amazonaws.com/mnist/"
FILES = {  # MNIST's standard MD5 hashes
    "train-images-idx3-ubyte.gz": "f68b3c2dcbeaaa9fbdd348bbdeb94873",
    "train-labels-idx1-ubyte.gz": "d53e105ee54ea40749a09fcbcd1e9432",
    "t10k-images-idx3-ubyte.gz": "9fb629c4189551a2d022fa330f9573f3",
    "t10k-labels-idx1-ubyte.gz": "ec29112dd5afa0611ce80d1b7f02629c",
}
DATA = Path("mnist")


def fetch():
    DATA.mkdir(exist_ok=True)
    for name, md5 in FILES.items():
        p = DATA / name
        if not p.exists():
            urllib.request.urlretrieve(URL + name, p)
        assert hashlib.md5(p.read_bytes()).hexdigest() == md5, f"{name} is corrupted; delete it and download it again"


def load(kind):
    """IDX format: the first 16 bytes of an image file (8 bytes of a label file) are the header; the rest are bytes
    0–255."""
    with gzip.open(DATA / f"{kind}-images-idx3-ubyte.gz") as f:
        images = np.frombuffer(f.read(), np.uint8, offset=16).reshape(-1, 28, 28)
    with gzip.open(DATA / f"{kind}-labels-idx1-ubyte.gz") as f:
        labels = np.frombuffer(f.read(), np.uint8, offset=8)
    return images, labels


fetch()
train_img, train_lab = load("train")
test_img, test_lab = load("t10k")
Xtr, ytr = train_img.reshape(-1, 784) / 255.0, train_lab.astype(np.int64)  # each row is one image
Xte, yte = test_img.reshape(-1, 784) / 255.0, test_lab.astype(np.int64)
print("training:", Xtr.shape, " test:", Xte.shape)

The course's network is MLSysBook's MLP: $h_1 = \mathrm{ReLU}(xW_1 + b_1)$, $h_2 = \mathrm{ReLU}(h_1W_2 + b_2)$,
$z = h_2W_3 + b_3$, then softmax and cross-entropy (Lesson 04). The cell below defines the forward pass and
initializes the network exactly as `reel/mnist_mlp/train.py` does (He initialization, seed 0), then shows it the
first training image.

The forward pass returns two things: the logits $z$, and the **intermediate values** $(x, h_1, h_2)$. At inference
time there's no need to keep them, but the backward pass will need them (sections 6 and 8).

In [ ]:
SIZES = [784, 128, 64, 10]


def init(seed=0):
    """He initialization as in train.py: W ~ N(0, 2/n_in), b = 0. Also returns rng: training keeps using it to shuffle
    the images."""
    rng = np.random.default_rng(seed)
    params = [
        [rng.normal(0, np.sqrt(2 / n_in), (n_in, n_out)), np.zeros(n_out)] for n_in, n_out in itertools.pairwise(SIZES)
    ]
    return params, rng


def softmax(z):
    e = np.exp(z - z.max(axis=1, keepdims=True))
    return e / e.sum(axis=1, keepdims=True)


def forward(params, x):
    """The forward pass for a batch x (one image per row): returns the logits z and the intermediate values the
    backward pass needs."""
    (W1, b1), (W2, b2), (W3, b3) = params
    h1 = np.maximum(0, x @ W1 + b1)
    h2 = np.maximum(0, h1 @ W2 + b2)
    return h2 @ W3 + b3, (x, h1, h2)


def cross_entropy(z, y):
    p = softmax(z)
    return -np.log(p[np.arange(len(y)), y]).mean()


params0, _ = init(0)
n_params = sum(t.size for layer in params0 for t in layer)
z0, _ = forward(params0, Xtr[:1])
p0 = softmax(z0)[0]
print(f"{num(n_params, 0)} parameters; the first image is a {ytr[0]}")
print(
    f"the untrained network gives the digit {ytr[0]} a probability of {pct(p0[ytr[0]], 1)}  →  loss −ln p = {num(-np.log(p0[ytr[0]]), 2)}"
)
untrained = cross_entropy(forward(params0, Xte)[0], yte)
print(
    f"mean loss on the 10,000 test images: {num(untrained, 4)}  (guessing all 10 digits equally: ln 10 = {num(np.log(10), 4)})"
)
assert n_params == 109386 and abs(untrained - 2.4257) < 1e-3, "differs from the reel and the demo: check the data"

The figure below shows that image (left) and the 10 probabilities that the untrained network assigns to the 10
digits (right). The bar of the correct answer is blue and hatched; the other bars are gray.

In [ ]:
fig, (a, b) = plt.subplots(1, 2, figsize=(9, 2.8), gridspec_kw={"width_ratios": [1, 3]})
a.imshow(train_img[0], cmap="gray_r")
a.set_title(f"training image #0: a {ytr[0]}", fontsize=10)
a.axis("off")
is_true = np.arange(10) == ytr[0]
b.bar(
    np.arange(10),
    p0,
    color=np.where(is_true, BLUE, GREY),
    hatch=["//" if t else "" for t in is_true],
    edgecolor="white",
)
b.set(xticks=range(10), xlabel="digit", ylabel="probability", title="Untrained network: the probability of each digit")
b.annotate("correct answer", (ytr[0], p0[ytr[0]]), xytext=(0, 4), textcoords="offset points", ha="center", fontsize=9)
fmt_axes(plt.gcf())
plt.show()

The untrained network can't tell the digits apart yet: it gives the 5 a probability of only 11.2%. That's why the mean
loss of 2.4257 is close to the 2.3026 of guessing all 10 digits equally. To lower the loss, gradient descent (Lesson 05) needs $\partial \mathcal{L} / \partial \theta$ for **each** of the 109,386 parameters.
“Whose fault” is this error? More precisely: if we nudge one parameter up a little, how much does the loss change?
That is a local sensitivity, at exactly the parameter's current value and for exactly the data at hand, not a
verdict on which parameter caused the error.

We start from a network small enough to compute by hand.

## 3 · The computational graph: each operation is a node

The lesson's demo uses a real but very small network: one input $x$, two ReLU neurons, one sigmoid neuron at the
output, and cross-entropy for a label $y \in \{0, 1\}$ (Lesson 04). It has 7 parameters $w_1, b_1, w_2, b_2, v_1, v_2, c$:

$$z_1 = w_1x + b_1, \qquad z_2 = w_2x + b_2, \qquad h_i = \mathrm{ReLU}(z_i)$$

$$z = v_1h_1 + v_2h_2 + c, \qquad \hat y = \sigma(z), \qquad \mathcal{L} = -\left[y \ln \hat y + (1 - y)\ln(1 - \hat y)\right]$$

Written as a **computational graph**, each operation is a node. The forward pass goes from $x$ down to $\mathcal{L}$
and keeps the value at every node. The cell below runs the forward pass with exactly the demo's starting values
($x = 1.5$ and the label $y = 1$).

In [ ]:
theta = {"w1": 0.6, "b1": -0.3, "w2": -0.4, "b2": 0.9, "v1": -0.5, "v2": 0.8, "c": -0.1}  # as in the demo
x_demo, y_demo = 1.5, 1


def softplus(a):
    return max(a, 0.0) + math.log1p(math.exp(-abs(a)))  # ln(1 + eᵃ), without overflowing


def forward_small(t, x, y):
    """The forward pass of the small graph: returns the value of every node (the backward pass will need them)."""
    v = {"x": x}
    v["z1"] = t["w1"] * x + t["b1"]
    v["z2"] = t["w2"] * x + t["b2"]
    v["h1"] = max(0.0, v["z1"])
    v["h2"] = max(0.0, v["z2"])
    v["z"] = t["v1"] * v["h1"] + t["v2"] * v["h2"] + t["c"]
    v["yh"] = 1 / (1 + math.exp(-v["z"]))
    v["L"] = y * softplus(-v["z"]) + (1 - y) * softplus(v["z"])  # = −[y ln ŷ + (1 − y) ln(1 − ŷ)], computed stably
    return v


NODE = {"x": "x", "z1": "z₁", "z2": "z₂", "h1": "h₁", "h2": "h₂", "z": "z", "yh": "ŷ", "L": "L"}
val = forward_small(theta, x_demo, y_demo)
print("  ·  ".join(f"{NODE[k]} = {num(val[k], 3)}" for k in ["z1", "z2", "h1", "h2", "z", "yh", "L"]))
print(f"the network gives a {pct(val['yh'], 1)} chance that the label is 1, so L = −ln ŷ = {num(val['L'], 3)}")
assert abs(val["L"] - 0.7763437730407396) < 1e-12, "differs from the demo: check the starting values"

Both ReLU neurons are open ($z_1, z_2 > 0$). The network gives $\hat y = 0.460$, that is, a 46.0% chance that the
label is 1, so the loss is $-\ln 0.460 = 0.776$.

## 4 · The chain rule runs backward through each node

To find $\partial \mathcal{L}/\partial w_1$, we go backward from $\mathcal{L}$ to $w_1$. The chain rule says that the
derivative of a composite function is the product of the derivatives of its parts:

$$\frac{\partial \mathcal{L}}{\partial w_1} = \frac{\partial \mathcal{L}}{\partial \hat y}\cdot\frac{\partial \hat y}{\partial z}\cdot\frac{\partial z}{\partial h_1}\cdot\frac{\partial h_1}{\partial z_1}\cdot\frac{\partial z_1}{\partial w_1}$$

Each factor is a **local derivative**: a node only needs to know its own operation and the values stored during the
forward pass. The backward pass receives the gradient from above, multiplies it by the local derivative, then passes
it on to the node below. This graph has three kinds of nodes, which CS231n calls patterns in backward flow:

- **Addition** copies the gradient unchanged to every one of its inputs: $\partial \mathcal{L}/\partial c = \partial \mathcal{L}/\partial z$.
- **Multiplication** swaps its two inputs: $\partial \mathcal{L}/\partial v_1 = \partial \mathcal{L}/\partial z \cdot h_1$,
  while $\partial \mathcal{L}/\partial h_1 = \partial \mathcal{L}/\partial z \cdot v_1$.
- **ReLU** is a gate: it multiplies by 1 if $z_i > 0$, and by 0 otherwise.

Sigmoid and cross-entropy together give a neat result: $\partial \mathcal{L}/\partial z = \hat y - y$.

### 🤔 Predict first
$x$ goes into both $z_1$ and $z_2$. So is $\partial \mathcal{L}/\partial x$ the gradient through branch 1, through
branch 2, or the sum of the two branches?

In [ ]:
def backward_small(t, v, y):
    """The backward pass: each line is the gradient from above × a local derivative."""
    g = {"L": 1.0}
    g["yh"] = -y / v["yh"] + (1 - y) / (1 - v["yh"])  # ∂L/∂ŷ
    g["z"] = g["yh"] * v["yh"] * (1 - v["yh"])  # × σ′(z) = ŷ(1 − ŷ); exactly ŷ − y
    g["v1"], g["v2"], g["c"] = g["z"] * v["h1"], g["z"] * v["h2"], g["z"]  # multiplication: swap; addition: copy
    g["h1"], g["h2"] = g["z"] * t["v1"], g["z"] * t["v2"]
    g["z1"] = g["h1"] * (v["z1"] > 0)  # ReLU: the gate is open (× 1) or closed (× 0)
    g["z2"] = g["h2"] * (v["z2"] > 0)
    g["w1"], g["b1"] = g["z1"] * v["x"], g["z1"]
    g["w2"], g["b2"] = g["z2"] * v["x"], g["z2"]
    g["x"] = g["z1"] * t["w1"] + g["z2"] * t["w2"]  # x is used twice: the gradients of the two branches add up
    return g


grad = backward_small(theta, val, y_demo)
print(f"∂L/∂ŷ = {num(grad['yh'], 3)}  ×  σ′(z) = {num(val['yh'] * (1 - val['yh']), 3)}  →  ∂L/∂z = {num(grad['z'], 4)}")
print(f"exactly ŷ − y = {num(val['yh'] - y_demo, 4)}")
print(
    f"∂L/∂x = {num(grad['z1'] * theta['w1'], 3)} (branch 1) + {num(grad['z2'] * theta['w2'], 3)} (branch 2) = {num(grad['x'], 3)}"
)
print("gradients of the 7 parameters:", "  ".join(f"{p} {num(grad[p], 3)}" for p in theta))
DEMO_GRAD = {  # the same numbers the demo on the lesson page shows
    "w1": 0.40493616341667427, "b1": 0.26995744227778284, "w2": -0.6478978614666788, "b2": -0.43193190764445255,
    "v1": -0.32394893073333936, "v2": -0.16197446536666968, "c": -0.5399148845555657,
}  # fmt: skip
assert all(abs(grad[p] - DEMO_GRAD[p]) < 1e-12 for p in theta), "differs from the demo"

It's the sum. $x$ is used twice, so nudging $x$ changes both $z_1$ and $z_2$, and the effects along the two paths add
up. The general rule: **when a value is used several times, the gradients flowing back to it add up**. That is why
autograd writes `grad += …` and not `grad = …` (section 5). In the MNIST network, each pixel feeds into all 128
neurons of layer 1, so its gradient is the sum over 128 paths.

The figure below draws the whole graph after both passes. Each box shows the node's value (top line, forward pass)
and its gradient $\partial L/\partial(\cdot)$ (bottom line, backward pass). The solid blue arrows are the forward
pass; the dashed orange arrows are the backward pass, labeled with the local derivative that the gradient gets
multiplied by when it crosses that edge.

In [ ]:
def draw_graph(ax, t, v, g):
    pos = {"x": (0, 0), "z1": (1, 0.75), "z2": (1, -0.75), "h1": (2, 0.75), "h2": (2, -0.75), "z": (3, 0)}
    pos |= {"yh": (4, 0), "L": (5, 0)}
    pos = {k: (1.6 * c, r) for k, (c, r) in pos.items()}
    W, H = 0.98, 0.62
    edges = [
        ("x", "z1", t["w1"]),
        ("x", "z2", t["w2"]),
        ("z1", "h1", float(v["z1"] > 0)),
        ("z2", "h2", float(v["z2"] > 0)),
    ]
    edges += [("h1", "z", t["v1"]), ("h2", "z", t["v2"]), ("z", "yh", v["yh"] * (1 - v["yh"])), ("yh", "L", g["yh"])]
    for a, b, d in edges:
        (xa, ya), (xb, yb) = pos[a], pos[b]
        x0, x1 = xa + W / 2, xb - W / 2
        ax.annotate("", (x1, yb + 0.07), (x0, ya + 0.07), arrowprops={"arrowstyle": "-|>", "color": BLUE, "lw": 1.4})
        ax.annotate(
            "",
            (x0, ya - 0.07),
            (x1, yb - 0.07),
            arrowprops={"arrowstyle": "-|>", "color": ORANGE, "lw": 1.4, "ls": "--"},
        )
        side = 1 if ya + yb > 0 else -1  # labels go on the outside: above the upper edges, below the lower ones
        ax.text(
            (x0 + x1) / 2,
            (ya + yb) / 2 + side * 0.26,
            f"×{num(d, 3)}",
            ha="center",
            va="center",
            fontsize=8,
            bbox={"fc": "white", "ec": "none", "pad": 0.5},
        )
    for k, (cx, cy) in pos.items():
        ax.add_patch(FancyBboxPatch((cx - W / 2, cy - H / 2), W, H, boxstyle="round,pad=0.02", fc="white", ec=GREY))
        ax.text(cx, cy + 0.14, f"{NODE[k]} = {num(v[k], 3)}", ha="center", va="center", fontsize=9, weight="bold")
        ax.text(cx, cy - 0.15, f"∂L/∂{NODE[k]} = {num(g[k], 3)}", ha="center", va="center", fontsize=8)
    ax.set(xlim=(-0.6, 5 * 1.6 + 0.6), ylim=(-1.25, 1.25))
    ax.axis("off")
    ax.legend(
        [Line2D([], [], color=BLUE, lw=1.6), Line2D([], [], color=ORANGE, lw=1.6, ls="--")],
        ["forward pass: values", "backward pass: gradient × local derivative"],
        loc="upper right", frameon=False, fontsize=8,
    )  # fmt: skip


fig, ax = plt.subplots(figsize=(10, 3.6))
draw_graph(ax, theta, val, grad)
ax.set_title(f"The demo's computational graph: x = {num(x_demo, 1)}, y = {y_demo}", fontsize=11)
fmt_axes(plt.gcf())
plt.show()

Reading the figure from right to left is reading the backward pass: $\partial L/\partial L = 1$, times $-2.174$ gives
$\partial L/\partial \hat y$, times $\sigma'(z) = 0.248$ gives $\partial L/\partial z = -0.540$, which then splits
into two branches. The upper branch multiplies by $v_1 = -0.5$, so the sign flips: $\partial L/\partial w_1 > 0$,
that is, increasing $w_1$ makes the error *grow*. Both ReLU gates are open (× 1).

### Checking by nudging
The gradient says: if we increase $w_1$ by $h$, then $L$ changes by about $\frac{\partial L}{\partial w_1} h$. We
check that just as the demo's “Try nudging” box does: increase each parameter by $h = 0.001$, run the forward pass
again, and measure $\Delta L$.

In [ ]:
h = 1e-3
passes = 0
for p, value in theta.items():
    nudged = dict(theta, **{p: value + h})
    dL = forward_small(nudged, x_demo, y_demo)["L"] - val["L"]
    passes += 1
    print(f"{p:>2}: measured ΔL {num(dL, 7):>11}   ∂L/∂{p}·h = {num(grad[p] * h, 7):>11}")
    assert abs(dL - grad[p] * h) < 1e-3 * abs(grad[p] * h)
print(
    f"that took {passes} extra forward passes, one per parameter; backpropagation gives all {len(theta)} gradients with just 1 backward pass"
)

The two columns agree to the third significant digit; the remaining discrepancy is there because $h$ is not
infinitely small (section 7 looks at this more closely). Nudging is easy to write and always right, but it costs
**one forward pass per parameter**: 7 passes here, 109,386 passes for the MNIST network. Backpropagation gives every
gradient with just one forward pass and one backward pass.

### One complete loop
Backpropagation only *computes* the gradient. *Using* it is the job of gradient descent (Lesson 05):
$\theta \leftarrow \theta - \eta\,\partial L/\partial\theta$, with $\eta = 0.5$ as in the demo.

In [ ]:
eta_demo = 0.5
theta_next = {p: theta[p] - eta_demo * grad[p] for p in theta}
L_next = forward_small(theta_next, x_demo, y_demo)["L"]
print("  ".join(f"{p} {num(theta[p], 2)} → {num(theta_next[p], 3)}" for p in theta))
print(f"loss: {num(val['L'], 3)} → {num(L_next, 3)}")
assert abs(L_next - 0.3137786399893179) < 1e-12, "differs from the demo"

One loop of forward → loss → backward → update brings the loss down from 0.776 to 0.314. Training the MNIST network
means repeating exactly this loop thousands of times; the only difference is that each node is a matrix.

## 5 · Automating it: a tiny autograd

`backward_small` is written by hand, line by line, for exactly this graph. Each node only needs to know its local
derivative, so the machine can do the rest by itself: during the forward computation, each value remembers which
values it was made from and what the local derivatives are; during the backward computation, it walks through the
nodes in reverse order and accumulates the gradients. That is the idea behind
[micrograd](https://github.com/karpathy/micrograd) (Karpathy), and behind autograd in PyTorch or JAX, at the scale of
tensors.

In [ ]:
class Value:
    """A scalar that remembers where it came from, so it can run the chain rule backward by itself (as in micrograd)."""

    def __init__(self, data, parents=(), local=()):
        self.data, self.grad = data, 0.0
        self.parents, self.local = parents, local  # local[i] = ∂(this node)/∂(parents[i]), saved in the forward pass

    def __add__(self, o):
        o = o if isinstance(o, Value) else Value(o)
        return Value(self.data + o.data, (self, o), (1.0, 1.0))  # addition: copies the gradient

    def __mul__(self, o):
        o = o if isinstance(o, Value) else Value(o)
        return Value(self.data * o.data, (self, o), (o.data, self.data))  # multiplication: swaps

    def relu(self):
        return Value(max(0.0, self.data), (self,), (float(self.data > 0),))

    def sigmoid(self):
        s = 1 / (1 + math.exp(-self.data))
        return Value(s, (self,), (s * (1 - s),))

    def log(self):
        return Value(math.log(self.data), (self,), (1 / self.data,))

    def backward(self):
        order, seen = [], set()

        def visit(v):  # topological order: a node comes after every node that produced it
            if id(v) not in seen:
                seen.add(id(v))
                for p in v.parents:
                    visit(p)
                order.append(v)

        visit(self)
        self.grad = 1.0
        for v in reversed(order):  # walk backward: by the time we reach v, every node using v has sent it its gradient
            for p, d in zip(v.parents, v.local):
                p.grad += v.grad * d  # +=: when a value is used several times, the gradients add up
        return len(order)

The cell below rebuilds exactly the demo's graph with `Value`. With the label $y = 1$, the loss is
$L = -\ln \hat y$. We don't write a single line of derivatives; we only call `L.backward()`.

In [ ]:
P = {p: Value(v) for p, v in theta.items()}
xv = Value(x_demo)
h1v = (P["w1"] * xv + P["b1"]).relu()
h2v = (P["w2"] * xv + P["b2"]).relu()
yhv = (P["v1"] * h1v + P["v2"] * h2v + P["c"]).sigmoid()
L = yhv.log() * -1.0  # y = 1
n_nodes = L.backward()
print(f"L = {num(L.data, 3)}; {n_nodes} nodes in the graph")
print("autograd:", "  ".join(f"{p} {num(P[p].grad, 3)}" for p in theta), f"  x {num(xv.grad, 3)}")
assert all(abs(P[p].grad - grad[p]) < 1e-12 for p in theta) and abs(xv.grad - grad["x"]) < 1e-12

The same 7 gradients (and $\partial L/\partial x$) as the hand-written version, exact up to rounding error. Two
details worth remembering:

- Each node keeps its local derivatives (that is, the intermediate values) from the forward pass until the backward
  pass goes through it. That is the memory that training has to pay on top of what prediction alone needs
  (section 8).
- `p.grad += …`: if you write `=`, $x$ keeps only the gradient of the branch processed last. This is a very common bug
  when writing an autograd yourself.

The idea is older than neural networks. Linnainmaa described the reverse mode of automatic differentiation in his
1970 thesis, to measure how rounding errors propagate through a long computation (Griewank, 2012). Werbos applied it
to neural networks in 1982. The 1986 paper by Rumelhart, Hinton & Williams showed that it lets multilayer networks
learn useful representations on their own, and made it popular.

## 6 · The backward pass in matrix form for the MNIST network

The 784 → 128 → 64 → 10 network is still a graph like this one; the only difference is that each node is now a
matrix. For a batch of $B$ images (each row of $X$ is one image), let $a_1 = XW_1 + b_1$, $h_1 = \mathrm{ReLU}(a_1)$,
$a_2 = h_1W_2 + b_2$, $h_2 = \mathrm{ReLU}(a_2)$, $z = h_2W_3 + b_3$, and let $\mathcal{L}$ be the batch's mean
cross-entropy.

The output: Lesson 04 gives the derivative of softmax + cross-entropy with respect to the logits as $p - y$ ($y$
one-hot). Since $\mathcal L$ is the mean over $B$ images, we also divide by $B$:

$$\frac{\partial \mathcal L}{\partial z} = \frac{1}{B}(P - Y)$$

After that, each layer $z = hW + b$ repeats the same three formulas (MLSysBook's equations 22–24), and ReLU
multiplies by a mask:

$$\frac{\partial \mathcal L}{\partial W} = h^\top \frac{\partial \mathcal L}{\partial z}, \qquad \frac{\partial \mathcal L}{\partial b} = \mathbf{1}^\top \frac{\partial \mathcal L}{\partial z}, \qquad \frac{\partial \mathcal L}{\partial h} = \frac{\partial \mathcal L}{\partial z}\, W^\top, \qquad \frac{\partial \mathcal L}{\partial a} = \frac{\partial \mathcal L}{\partial h} \odot [a > 0]$$

This is still section 4's swapping multiplication, written for matrices: the gradient of $W$ uses the stored *input*
$h$, and the gradient of $h$ uses the *weights* $W$. $\mathbf 1^\top(\cdot)$ sums the rows: $b$ is used for all $B$
images, so the gradients of the $B$ images add up. There's no need to memorize where the transposes go:
$\partial\mathcal L/\partial W$ must have exactly the shape of $W$, and only one way of multiplying gives that shape
(CS231n calls this dimension analysis).

The cell below writes the backward pass exactly as in `train.py`, runs it on the first 64 training images and prints
the shape of each gradient. The `relu_gate` argument is only there so that section 7 can introduce a bug on purpose.

In [ ]:
def backward(params, cache, z, y, relu_gate=True):
    """The backward pass of the MNIST network: the gradient of the mean cross-entropy with respect to all 6 parameter
    tensors."""
    x, h1, h2 = cache  # the intermediate values the forward pass kept
    _, (W2, _), (W3, _) = params
    dz = softmax(z)
    dz[np.arange(len(y)), y] -= 1  # ∂L/∂z = p − y (Lesson 04) …
    dz /= len(y)  # … divided by B because L is the mean over the batch
    da2 = dz @ W3.T  # ∂L/∂h₂
    if relu_gate:
        da2 = da2 * (h2 > 0)  # through the ReLU gate: ∂L/∂a₂
    da1 = da2 @ W2.T  # ∂L/∂h₁
    if relu_gate:
        da1 = da1 * (h1 > 0)  # ∂L/∂a₁
    return [[x.T @ da1, da1.sum(0)], [h1.T @ da2, da2.sum(0)], [h2.T @ dz, dz.sum(0)]]


NAMES = ["W₁", "b₁", "W₂", "b₂", "W₃", "b₃"]


def tensors(params):
    return [t for layer in params for t in layer]


xb, yb = Xtr[:64], ytr[:64]
zb, cache = forward(params0, xb)
grads = backward(params0, cache, zb, yb)
for name, P, G in zip(NAMES, tensors(params0), tensors(grads)):
    print(f"{name}: parameter {P.shape!s:>10}   gradient {G.shape!s:>10}")
    assert P.shape == G.shape

Each gradient has exactly the shape of its parameter: 109,386 numbers, after one forward pass and one backward pass.

### Summing over every path
The formula on the lesson page is written for a weight that feeds straight into the output. For a weight of layer 1,
say $W_1[i, j]$, which connects pixel $i$ to hidden neuron $j$, its influence reaches $\mathcal L$ along a great many
paths: through each image $n$ of the batch, each neuron $m$ of layer 2, each logit $k$. The chain rule gives each path
a product of local derivatives, and the gradient is their sum:

$$\frac{\partial \mathcal L}{\partial W_1[i,j]} = \sum_{n}\sum_{m}\sum_{k} x_{ni}\,[a_{1,nj} > 0]\,W_2[j,m]\,[a_{2,nm} > 0]\,W_3[m,k]\,\frac{\partial \mathcal L}{\partial z_{nk}}$$

The cell below adds up exactly that sum with three nested loops, for the layer-1 weight with the largest gradient,
then compares it with the backward pass.

In [ ]:
x0, h1, h2 = cache
dz = (softmax(zb) - np.eye(10)[yb]) / len(yb)
(W1, _), (W2, _), (W3, _) = params0
i, j = np.unravel_index(np.abs(grads[0][0]).argmax(), W1.shape)
total, n_paths, n_open = 0.0, 0, 0
for n in range(len(yb)):
    for m in range(64):
        for k in range(10):
            term = x0[n, i] * (h1[n, j] > 0) * W2[j, m] * (h2[n, m] > 0) * W3[m, k] * dz[n, k]
            n_paths += 1
            n_open += term != 0
            total += term
print(
    f"W₁[{i}, {j}]: a sum over {num(n_paths, 0)} paths (64 images × 64 layer-2 neurons × 10 logits), {num(n_open, 0)} of them nonzero"
)
print(f"loops: {num(total, 6)}   backward pass: {num(grads[0][0][i, j], 6)}")
assert np.isclose(total, grads[0][0][i, j], rtol=1e-12, atol=0)

The two ways give the same number. A path that goes through an image with no stroke at pixel $i$, or through a
closed ReLU gate, contributes exactly 0. Listing every path for every weight is far too expensive: this one weight
alone has 40,960 paths. The backward pass doesn't list them: it computes $\partial\mathcal L/\partial h_2$ and then
$\partial\mathcal L/\partial h_1$ **once**, and every weight further back reuses those results. The part that all the
paths share is computed once; that is where the savings come from.

## 7 · Checking each tensor with finite differences

A hand-written backward pass can be wrong and still run, still produce the right shapes. The standard check is to
compare it with the *numerical* gradient, measured directly by nudging as in section 4, but with the central
difference, which is much more accurate:

$$\frac{\partial \mathcal L}{\partial \theta_i} \approx \frac{\mathcal L(\theta_i + h) - \mathcal L(\theta_i - h)}{2h}$$

We compare the two numbers with the **relative error** $|a - n| / \max(|a|, |n|)$, because gradients can be very
small or very large. According to CS231n, below $10^{-7}$ is good; for functions with kinks, like ReLU, below
$10^{-4}$ is usually still fine; above $10^{-2}$ it is probably wrong. CS231n also advises checking a few elements of
**every** tensor, biases included.

The cell below checks all 6 tensors on section 6's batch of 64 images. Small tensors (≤ 1,000 elements) are checked
in full. For $W_1$ and $W_2$, we check 300 randomly chosen elements. Each element costs two forward passes.

In [ ]:
def loss_of(params, x, y):
    return cross_entropy(forward(params, x)[0], y)


def numerical(params, x, y, t, idx, h=1e-5):
    """The numerical gradient of element idx of the t-th tensor: (L(θ + h) − L(θ − h)) / 2h."""
    P = tensors(params)[t]
    old = P[idx]
    P[idx] = old + h
    L_plus = loss_of(params, x, y)
    P[idx] = old - h
    L_minus = loss_of(params, x, y)
    P[idx] = old  # put the old value back
    return (L_plus - L_minus) / (2 * h)


def rel_err(a, n):
    d = max(abs(a), abs(n))
    return 0.0 if d == 0 else abs(a - n) / d


rng_check = np.random.default_rng(1)
checks = []  # for each tensor: a list of (index, numerical gradient)
for t, P in enumerate(tensors(params0)):
    if P.size <= 1000:
        idxs = list(np.ndindex(P.shape))
    else:
        idxs = [np.unravel_index(f, P.shape) for f in rng_check.choice(P.size, 300, replace=False)]
    checks.append([(i, numerical(params0, xb, yb, t, i)) for i in idxs])


def worst_errors(grads):
    return [max(rel_err(tensors(grads)[t][i], n) for i, n in chk) for t, chk in enumerate(checks)]


worst = worst_errors(grads)
for name, chk, e in zip(NAMES, checks, worst):
    print(f"{name}: checked {num(len(chk), 0):>3} elements, largest relative error {sci(e)}")
print(f"{num(sum(map(len, checks)), 0)} elements in total, {num(2 * sum(map(len, checks)), 0)} forward passes")
assert max(worst) < 1e-4, "the backward pass doesn't match the numerical gradient"

All 6 tensors match, with the largest error around $10^{-6}$ or below. The remaining discrepancy comes from rounding,
when the two values $\mathcal L(\theta \pm h)$ are almost equal, especially for elements with very small gradients.

### 🤔 Predict first
A very common bug: forgetting to multiply by the ReLU gate in the backward pass (calling
`backward(..., relu_gate=False)`). Which of the 6 tensors will fail the check: all of them, only the tensors near the
input, or only the tensors near the output?

In [ ]:
variants = {
    "correct": grads,
    "forgot the ReLU gate": backward(params0, cache, zb, yb, relu_gate=False),
    "forgot to divide by B": [[len(yb) * g for g in layer] for layer in grads],  # the gradient of the sum, not the mean
}
print(f"{'':>21}" + "".join(f"{n:>10}" for n in NAMES))
table = {}
for label, g in variants.items():
    table[label] = worst_errors(g)
    print(f"{label:>21}" + "".join(f"{sci(e) if e < 0.01 else num(e, 2):>10}" for e in table[label]))
gate = table["forgot the ReLU gate"]
assert max(gate[4:]) < 1e-4 and min(gate[:4]) > 1e-2
assert all(abs(e - 63 / 64) < 1e-3 for e in table["forgot to divide by B"])

Forgetting the ReLU gate breaks only $W_1, b_1, W_2, b_2$. $W_3$ and $b_3$ still pass the check, because their
gradients are computed *before* the backward pass goes through any ReLU gate. The backward pass goes from the output
back to the input, so a bug at one node only breaks what lies before it (closer to the input). If we checked only the
last layer, we would not see this bug. Forgetting to divide by $B$ makes every gradient 64 times too large: a relative
error of $63/64 \approx 0.98$ in all 6 tensors.

### Choosing the nudge $h$
Is a large $h$ better, or a small one? The cell below tries $h$ from $10^{-1}$ to $10^{-12}$ on the weight of $W_2$
with the largest gradient, in float64 (numpy's default) and in float32 (the number type GPUs usually use). It also
counts how many of layer 2's ReLU gates change state between $\theta + h$ and $\theta - h$ over the 64 images.

In [ ]:
k2 = np.unravel_index(np.abs(grads[1][0]).argmax(), W2.shape)  # the weight of W₂ with the largest gradient
a2 = grads[1][0][k2]
params32 = [[t.astype(np.float32) for t in layer] for layer in params0]
xb32 = xb.astype(np.float32)


def gate_flips(params, x, t, idx, h):
    """The number of layer-2 ReLU gates (64 images × 64 neurons) whose state differs between θ + h and θ − h."""
    P = tensors(params)[t]
    old = P[idx]
    P[idx] = old + h
    plus = forward(params, x)[1][2] > 0
    P[idx] = old - h
    minus = forward(params, x)[1][2] > 0
    P[idx] = old
    return int((plus != minus).sum())


hs = 10.0 ** -np.arange(1, 12.5, 0.5)
e64 = np.array([rel_err(a2, numerical(params0, xb, yb, 2, k2, h)) for h in hs])
e32 = np.array([rel_err(a2, numerical(params32, xb32, yb, 2, k2, h)) for h in hs])
nflip = np.array([gate_flips(params0, xb, 2, k2, h) for h in hs])
print(f"W₂[{k2[0]}, {k2[1]}]: gradient from the backward pass {num(a2, 6)}")
for h, a, b, f in zip(hs[::2], e64[::2], e32[::2], nflip[::2]):
    print(f"h = {sci(h, 0):>7}: error float64 {sci(a):>9}   float32 {sci(b):>9}   ReLU gates that change state: {f}")
best = e64.argmin()
print(f"best: float64 {sci(e64[best])} at h = {sci(hs[best])}; float32 never below {sci(e32.min())}")
i5 = np.abs(np.log10(hs) + 5).argmin()  # h = 10⁻⁵, the step section 7 uses
assert e64[i5] < 1e-8 and e32.min() > 1e-4 and nflip[0] > 0

The figure below plots the same numbers: the horizontal axis is $h$, the vertical axis is the relative error (both on
a log scale). The blue line with round dots is float64; the dashed orange line with square dots is float32. The gray
region covers the values of $h$ for which at least one ReLU gate changes state between $\theta + h$ and
$\theta - h$. The two horizontal dotted lines are CS231n's thresholds, $10^{-7}$ and $10^{-2}$.

In [ ]:
fig, ax = plt.subplots(figsize=(8.5, 3.8))
ax.axvspan(hs[nflip > 0].min() / 10**0.25, hs.max() * 10**0.25, color=GREY, alpha=0.18, lw=0)
ax.text(hs.max() / 1.2, 3e-12, "ReLU gates\nchange state", ha="right", va="bottom", fontsize=9)
ax.loglog(hs, e64, "o-", color=BLUE, ms=5, label="float64")
ax.loglog(hs, e32, "s--", color=ORANGE, ms=5, label="float32")
for level, note in [(1e-7, "10⁻⁷: good"), (1e-2, "10⁻²: probably wrong")]:
    ax.axhline(level, color=GREY, lw=1, ls=":")
    ax.text(hs.min(), level * 1.6, note, fontsize=8, color="#444444")
ax.set(
    xlabel="nudge h",
    ylabel="relative error",
    ylim=(1e-12, 3),
    title="Numerical gradient against the backward pass, one weight of W₂",
)
ax.legend(frameon=False, loc="center left")
fmt_axes(plt.gcf())
plt.show()

There are three regions:

- **Large $h$** (the gray region): somewhere in the interval $[\theta - h, \theta + h]$ a ReLU gate closes or opens,
  so the loss function has a kink there. The difference measures the average slope across the kink, so it's off by
  around $10^{-3}$ to $10^{-2}$ even though the backward pass is entirely correct.
- **Moderate $h$** (around $10^{-5}$): float64 agrees to about $10^{-10}$.
- **$h$ too small:** $\mathcal L(\theta + h)$ and $\mathcal L(\theta - h)$ are almost equal, and the subtraction loses
  almost all of the significant digits, so the error grows again.

float32 keeps only about 7 significant digits (float64 about 16), so at no step does it do better than 9.3·10⁻³,
right at the $10^{-2}$ threshold that CS231n considers “probably wrong”. That is why gradient checks always run in
float64, with $h$ around $10^{-5}$, on few images (fewer images, fewer kinks).

## 8 · The cost: computation and memory

### 🤔 Predict first
The network's forward pass costs 109,184 multiply-accumulate (MAC) operations per image (Lesson 01). The backward
pass produces 109,386 gradients. How many MACs does it cost: fewer than the forward pass, about the same, about twice
as many, or hundreds of thousands of times as many?

In [ ]:
fwd = bwd = 0
print(f"{'layer':>11}{'forward: hW':>13}{'backward ∂W = hᵀ·∂z':>22}{'backward ∂h = ∂z·Wᵀ':>22}")
for layer, (n_in, n_out) in enumerate(itertools.pairwise(SIZES), 1):
    mac = n_in * n_out  # MACs per image of one multiplication by an n_in × n_out matrix
    mac_h = 0 if layer == 1 else mac  # layer 1: nobody needs the gradient with respect to the pixels x
    fwd, bwd = fwd + mac, bwd + mac + mac_h
    print(f"{f'{n_in} → {n_out}':>11}{num(mac, 0):>13}{num(mac, 0):>22}{num(mac_h, 0):>22}")
print(
    f"per image: forward {num(fwd, 0)} MACs, backward {num(bwd, 0)} MACs = {num(bwd / fwd, 2)} times the forward pass"
)
print(f"one training step (forward + backward) = {num((fwd + bwd) / fwd, 2)} times the forward pass")
nudge = n_params * fwd  # one-sided nudging: one forward pass per parameter
print(
    f"nudging each parameter: {num(n_params, 0)} forward passes = {num(nudge, 0)} MACs, {num(nudge / bwd, 0)} times the backward pass"
)
assert (fwd, bwd) == (109184, 118016)

The backward pass costs only 1.08 times the forward pass. Each layer needs two matrix multiplications of the same
size as in the forward pass, one for $\partial W$ and one for $\partial h$, so for dense layers the backward pass
usually costs about twice as much (MLSysBook). In this network, the first layer accounts for 100,352 of the 109,184
MACs, and that layer doesn't need $\partial\mathcal L/\partial x$ (nobody learns the pixels), so the ratio drops to
just 1.08. Adding in the bias additions and the ReLU gates doesn't change it much either: they cost only about
$B \cdot n$ operations per layer.

What matters is that this ratio is a **constant**: the backward pass costs about as much as the forward pass,
however many parameters the network has. Nudging each parameter in turn is more expensive by a factor equal to the
number of parameters. The cell below measures the real time on the machine you are running on (the numbers depend
on the machine; they are usually close to the MAC ratio, but not exactly equal to it).

In [ ]:
def timeit(f, repeat=200):
    f()
    t0 = time.perf_counter()
    for _ in range(repeat):
        f()
    return (time.perf_counter() - t0) / repeat


t_f = timeit(lambda: forward(params0, xb))
t_b = timeit(lambda: backward(params0, cache, zb, yb))
print(
    f"a batch of 64 images on this machine: forward {num(t_f * 1e3, 3)} ms, backward {num(t_b * 1e3, 3)} ms ({num(t_b / t_f, 1)} times as long)"
)
print(
    f"nudging all {num(n_params, 0)} parameters: about {num(n_params * t_f, 0)} seconds for ONE gradient descent step"
)

### Memory: the real cost of the backward pass
The backward pass needs $x, h_1, h_2$ and $z$ from the forward pass (the formula $\partial W = h^\top \partial z$ uses
the layer's input $h$, the ReLU mask uses $h > 0$). So during training, every intermediate value of the whole batch
must stay in memory until the backward pass has gone through its layer. At inference, once a layer is computed, its
input can be discarded. Training also has to keep the gradients (the same size as the parameters) and the optimizer's
state: Adam (Lesson 07) keeps two more numbers per parameter.

The cell below rebuilds MLSysBook's simplified memory ledger (Table 8: a batch of 32 images, 4 bytes per number in
FP32, Adam) for exactly this network. As in the book, 1 kB = 1,000 bytes.

In [ ]:
def ledger(B, sizes=SIZES, adam=True, nbytes=4):
    """MLSysBook's simplified memory ledger: the bytes of each part during training and at inference."""
    n = sum(a * b + b for a, b in itertools.pairwise(sizes))  # the number of parameters
    saved = B * sum(sizes)  # x, h₁, h₂, z of the whole batch: kept from the forward pass until the backward pass
    peak = B * max(a + b for a, b in itertools.pairwise(sizes))  # inference: only the current layer's input and output
    train = {"parameters": n, "intermediate values": saved, "gradients": n, "Adam state": 2 * n * adam}
    infer = {"parameters": n, "intermediate values": peak}
    return {k: v * nbytes for k, v in train.items()}, {k: v * nbytes for k, v in infer.items()}


mem_train, mem_infer = ledger(32)
print(
    f"intermediate values to keep during training, batch of 32: 32 × {sum(SIZES)} = {num(32 * sum(SIZES), 0)} numbers"
)
for part, nb in mem_train.items():
    print(f"{part:>19}: training {num(nb / 1000, 1):>6} kB   inference {num(mem_infer.get(part, 0) / 1000, 1):>6} kB")
ratio = sum(mem_train.values()) / sum(mem_infer.values())
print(
    f"{'total':>19}: training {num(sum(mem_train.values()) / 1e6, 1)} MB, inference {num(sum(mem_infer.values()) / 1000, 0)} kB"
    f"  →  {num(ratio, 1)} times as much"
)
assert 32 * sum(SIZES) == 31552 and round(ratio, 1) == 3.4, "differs from MLSysBook's Table 8"

Training takes 3.4 times the memory of inference alone, exactly as in MLSysBook's Table 8. The figure below draws the
same ledger as two bars. Each part has its own color and hatch pattern, and its name is written right on the bar.

In [ ]:
parts = [
    ("parameters", BLUE, ""),
    ("intermediate values", ORANGE, "//"),
    ("gradients", GREEN, "\\\\"),
    ("Adam state", GREY, ".."),
]
fig, ax = plt.subplots(figsize=(9, 2.4))
for row, mem in enumerate([mem_infer, mem_train]):
    left = 0
    for part, color, hatch in parts:
        kb = mem.get(part, 0) / 1000
        if kb:
            ax.barh(row, kb, left=left, color=color, hatch=hatch, edgecolor="white", height=0.6)
            name = textwrap.fill(part, 12)  # a long name wraps, so it fits in its segment
            ax.text(left + kb / 2, row, f"{name}\n{num(kb, 0)} kB", ha="center", va="center", fontsize=8)
            left += kb
    ax.text(left + 40, row, f"{num(left, 0)} kB", va="center", fontsize=9, weight="bold")
ax.set(yticks=[0, 1], yticklabels=["inference", "training"], xlabel="kB (FP32, batch of 32 images)", xlim=(0, 2150))
ax.set_title("Memory of the 784 → 128 → 64 → 10 network: training against inference alone", fontsize=11)
fmt_axes(plt.gcf())
plt.show()

The backward pass doesn't take many more operations; what it really demands is **memory**. For this small network,
most of the memory goes to the parameters, the gradients and the Adam state. With large batches, the intermediate
values take up the most: they grow with the batch size and with the *total* width of all the layers, whereas
inference only needs the layer being computed (section 10 tries this). When memory runs short, one can drop some of
the intermediate values and recompute them when the backward pass needs them (*activation checkpointing*, Chen et
al., 2016): trading extra computation for memory.

## 9 · Training with your own backward pass

Now we put the four steps forward → loss → backward → update together into the training loop of
`reel/mnist_mlp/train.py`: He initialization (seed 0), mini-batch SGD with $\eta = 0.1$, batches of 64 images,
5 epochs over the 60,000 images, shuffling the images with the very `rng` created at initialization. The loop records
the mean loss after every 50 steps and the test accuracy after each epoch.

In [ ]:
def train(seed=0, epochs=5, eta=0.1, batch=64, relu_gate=True, log_every=50):
    params, rng = init(seed)
    history, test_acc, running, step = [], [], [], 0
    for _ in range(epochs):
        order = rng.permutation(len(Xtr))
        for i in range(0, len(order), batch):
            idx = order[i : i + batch]
            z, cache = forward(params, Xtr[idx])  # 1. forward pass, keeping the intermediate values
            running.append(cross_entropy(z, ytr[idx]))  # 2. loss
            grads = backward(params, cache, z, ytr[idx], relu_gate)  # 3. backward pass
            for P, G in zip(tensors(params), tensors(grads)):  # 4. update: gradient descent (Lesson 05)
                P -= eta * G
            step += 1
            if step % log_every == 0:
                history.append(np.mean(running))
                running = []
        test_acc.append((forward(params, Xte)[0].argmax(1) == yte).mean())
    return params, np.array(history), np.array(test_acc)


t0 = time.perf_counter()
params, history, test_acc = train()
print(f"5 epochs, {num(5 * math.ceil(len(Xtr) / 64), 0)} updates in {num(time.perf_counter() - t0, 1)} seconds")
print("test accuracy after each epoch:", " → ".join(pct(a) for a in test_acc))
z_te = forward(params, Xte)[0]
test_loss = cross_entropy(z_te, yte)
train_acc = (forward(params, Xtr)[0].argmax(1) == ytr).mean()
print(f"training loss: {num(history[0], 4)} (first 50 steps) → {num(history[-1], 4)} (last 50 steps)")
print(f"test loss {num(test_loss, 4)}  ·  {pct(train_acc)} of the training images correct")
REF = [0.9427, 0.9633, 0.9688, 0.9706, 0.9727]  # epochLog of mnist-mlp.json (reel/mnist_mlp/train.py)
assert all(abs(a - r) < 5e-4 for a, r in zip(test_acc, REF)), "differs from the reel and the demo: check the data"
assert abs(test_loss - 0.0865) < 5e-4 and abs(train_acc - 0.9828) < 5e-4
assert abs(history[0] - 1.2031) < 1e-3 and abs(history[-1] - 0.059) < 1e-3

After 5 epochs, the network gets **97.27%** of the 10,000 test images right, with a loss of 0.0865: exactly the
numbers of the reel and the demo of Lesson 04, because this is the same computation with the same seed. The cell
below compares all 109,386 weights with the reference copy that `train.py` produces. On the machine that built this
notebook, they match bit for bit. On a different CPU, the BLAS library may round the last digit differently: the line
below then prints `False`, while every rounded number above stays the same.

In [ ]:
md5 = hashlib.md5(b"".join(t.tobytes() for t in tensors(params))).hexdigest()
print("bit-for-bit identical to train.py's weights:", md5 == "e61017ca6e9fe52bc61dd8079ac77cd9")

The figure below shows the training loss (averaged over every 50 steps) against the number of updates. The dotted
vertical lines mark the end of each epoch, labeled with the test accuracy at that point.

In [ ]:
steps = 50 * np.arange(1, len(history) + 1)
per_epoch = math.ceil(len(Xtr) / 64)
fig, ax = plt.subplots(figsize=(9, 3.4))
ax.plot(steps, history, color=BLUE, lw=2)
for e, a in enumerate(test_acc, 1):
    ax.axvline(e * per_epoch, color=GREY, lw=1, ls=":")
    ax.text(e * per_epoch - 40, 1.15, f"epoch {e}\n{pct(a)}", ha="right", va="top", fontsize=8)
ax.set(xlabel="update step (batch of 64 images)", ylabel="training loss", ylim=(0, 1.25), xlim=(0, 5 * per_epoch + 50))
ax.set_title("Training with our own backward pass (η = 0.1, seed 0)", fontsize=11)
fmt_axes(plt.gcf())
plt.show()

The loss falls fast during the first few hundred steps (from 1.2031 in the first 50 steps), then more and more slowly,
down to 0.059. Each point on the curve is an average over 50 batches, so it is still bumpy: each batch gives a
slightly different gradient.

### 🤔 Predict first
Each row of $W_1$ belongs to one pixel: row $i$ holds the 128 weights that connect pixel $i$ to the first hidden
layer. After 5 epochs and 4,690 updates, is any row still **exactly equal** to its value at initialization? If so, how
many rows, and why?

In [ ]:
params_init, _ = init(0)
unchanged = (params[0][0] == params_init[0][0]).all(axis=1)  # rows of W₁ that haven't changed by a single bit
blank = Xtr.max(axis=0) == 0  # pixels that are blank in all 60,000 training images
print(
    f"{unchanged.sum()} rows of W₁ unchanged; {blank.sum()} pixels blank in every training image; the same pixels: {np.array_equal(unchanged, blank)}"
)
one = backward(params_init, *forward(params_init, Xtr[:1])[::-1], ytr[:1])[0][0]
print(
    f"for image #0 alone: {(Xtr[0] == 0).sum()} of the 784 rows of ∂L/∂W₁ are 0 (pixels with no stroke), "
    f"and {(one == 0).all(axis=0).sum()} of the 128 columns are 0 (neurons whose ReLU gate is closed)"
)
assert unchanged.sum() == 67 and np.array_equal(unchanged, blank)

Exactly 67 rows haven't changed by a single bit, and they are exactly the 67 pixels that are blank in *every* training
image (mostly in the corners). The gradient of $W_1$ is $x^\top\,\partial\mathcal L/\partial a_1$, so row $i$ is
always multiplied by $x_i$: for a pixel that never has a stroke, the gradient of the whole row is always exactly 0,
and gradient descent never touches it. This is the same reason why 297 of the perceptron's weights in Lesson 01 stay
at 0.

For a single image, the gradient is much sparser still: the rows of the pixels that are blank in that image, and the
columns of the neurons whose ReLU gate is closed, are all 0. Backpropagation assigns responsibility only to what
actually took part in the forward pass.

### A bug that breaks nothing, but slows everything down
In section 7, forgetting the ReLU gate made 4 of the 6 tensors wrong. But it doesn't make the program report an error.
The cell below trains for 1 epoch with the correct backward pass and with the backward pass that lacks the ReLU gate,
3 seeds each.

In [ ]:
for gate in [True, False]:
    accs = [train(seed, epochs=1, relu_gate=gate)[2][-1] for seed in range(3)]
    print(
        f"{'correct backward pass' if gate else 'forgot the ReLU gate':>21}: after 1 epoch {pct(min(accs))} to {pct(max(accs))} correct (3 seeds)"
    )
    if gate:
        good = accs
assert min(good) > max(accs)

The network with the wrong gradients *still learns*: it still goes downhill, in a roughly right direction. But after
1 epoch it is about 3 percentage points behind the correct backward pass, far outside the variation between seeds.
There is no error message; only a model that is worse for no obvious reason. That is why you must check the gradient
of every tensor whenever you write a backward pass yourself: watching the loss curve go down is not enough. Karpathy
calls backpropagation a “leaky abstraction”: even when you use PyTorch, understanding it helps you see why a network
learns poorly.

## 10 · Try changing things

The cells below rerun the same functions with different arguments. You can edit the values and run them again.

### Memory by batch size and network size
The `ledger()` function from section 8 takes the batch size `B` and the layers `sizes`. Try larger batches, and a
larger network (4 hidden layers of 512):

In [ ]:
RUNS = [(32, SIZES), (256, SIZES), (4096, SIZES), (256, [784, 512, 512, 512, 512, 10])]
NET_W = max(len(" → ".join(map(str, sizes))) for _, sizes in RUNS)  # the network column's width, so the rows line up
for B, sizes in RUNS:
    tr, inf = ledger(B, sizes)
    act = tr["intermediate values"] / sum(tr.values())
    print(
        f"batch {B:>4}, network {' → '.join(map(str, sizes)):>{NET_W}}: training {num(sum(tr.values()) / 1e6, 1):>5} MB, "
        f"inference {num(sum(inf.values()) / 1e6, 2):>5} MB ({num(sum(tr.values()) / sum(inf.values()), 1):>4} times as much); "
        f"intermediate values take {pct(act, 0)} of the training memory"
    )

As the batch grows, the intermediate values go from 7% to 90% of the training memory. The ratio of training to
inference, on the other hand, *falls* (3.4 → 2.0 → 1.2 times), because inference also has to hold the whole batch,
if only for the layer being computed. With the network of 4 hidden layers of 512, the parameters, gradients and Adam
state take up most of the memory again. So the 3.4-fold figure is not a constant: it depends on the batch size, the
network size and the optimizer, as MLSysBook points out.

### The gradient of a batch against that of the whole training set
Gradient descent in Lesson 05 uses the gradient of a batch of 64 images in place of the gradient of all 60,000 images.
The backward pass can compute both. The cell below measures the cosine similarity between them (1 means the same
direction) for 20 random batches, with the freshly initialized network.

In [ ]:
def flat(g):
    return np.concatenate([t.ravel() for t in tensors(g)])


z_all, cache_all = forward(params_init, Xtr)
g_full = flat(backward(params_init, cache_all, z_all, ytr))
rng_b = np.random.default_rng(5)
cos = []
for _ in range(20):
    idx = rng_b.choice(len(Xtr), 64, replace=False)
    zb_, cb_ = forward(params_init, Xtr[idx])
    g = flat(backward(params_init, cb_, zb_, ytr[idx]))
    cos.append(g @ g_full / np.linalg.norm(g) / np.linalg.norm(g_full))
print(
    f"cosine between the gradient of a 64-image batch and that of all 60,000 images: {num(min(cos), 2)} to {num(max(cos), 2)} (20 batches)"
)
nz = np.abs(g_full[g_full != 0])
lo, hi = np.percentile(nz, [1, 99])
print(
    f"nonzero magnitudes |∂L/∂θ|: from {sci(lo)} (1st percentile) to {sci(hi)} (99th percentile), a {sci(hi / lo)}-fold spread"
)

The gradient of a batch points only *roughly* in the direction of the true gradient, and the gradients of different
parameters differ in magnitude by a factor of hundreds of thousands. Using the same $\eta$ for every parameter is
therefore a crude choice. That is where there is room to improve: in the update step, not in the backward pass.

### Seeds and batch size
Try other seeds, or smaller batches: the backward pass stays the same; only the number of steps and the noise of the
gradient change.

In [ ]:
for seed, batch in [(1, 64), (2, 64), (0, 16)]:
    t0 = time.perf_counter()
    acc = train(seed, epochs=1, batch=batch)[2][-1]
    print(
        f"seed {seed}, batch {batch:>2}: after 1 epoch {pct(acc)} correct ({num(time.perf_counter() - t0, 1)} seconds)"
    )

Seeds 1 and 2 fall within section 9's range of 94.17–94.86%. Batches of 16 images give 4 times as many updates per
epoch, so after 1 epoch the network already reaches 96.37%, at the price of calling the backward pass 4 times as
often, with a noisier gradient at each step.

## 11 · Limits and next steps

Backpropagation has done its job: correct gradients (checked tensor by tensor) for all 109,386 parameters, at roughly
the cost of one forward pass. What is still crude is the **update step**: SGD subtracts the same
$\eta \cdot \partial\mathcal L/\partial\theta$ from every parameter, even though the batch gradient is noisy and the
parameters' gradients differ enormously in size. Lesson 07 replaces that step with optimizers that remember the
history of the gradients: momentum smooths the direction of travel, and Adam divides each parameter's gradient by its
usual magnitude. The price is more memory: exactly two numbers per parameter, the “Adam state” part of section 8's
ledger.

One more limit: the backward pass goes through every layer, so the gradient of the first layer is a product of a
great many local derivatives. In a deep network, that product can shrink to 0 or blow up:
[Lesson 09 · Vanishing gradients & initialization](https://d3qr1cjln6qqtk.cloudfront.net/en/lessons/vanishing-gradient/).

## Summary
- The forward pass computes and **keeps** the intermediate values; the backward pass goes back from $\mathcal L$,
  multiplying the gradient from above by the local derivative of each node. The 7-parameter demo: $\mathcal L$ =
  0.776, and 0.314 after one update.
- Addition copies the gradient, multiplication swaps, ReLU is a gate; when a value is used several times, its
  gradients **add up**.
- Matrix form: $\partial W = h^\top \partial z$, $\partial b = \mathbf 1^\top \partial z$, $\partial h = \partial z\, W^\top$,
  starting from $\partial z = (P - Y)/B$.
- Check with central differences, in float64, with $h$ around $10^{-5}$, on every tensor: the largest error is around
  $10^{-6}$. Forgetting the ReLU gate still passes the check at $W_3$, $b_3$, and the network still learns, only about
  3 percentage points worse after 1 epoch.
- The backward pass costs 118,016 MACs against the forward pass's 109,184 (1.08 times); nudging needs 109,386 forward
  passes. The real cost is memory: with a batch of 32 images, training needs 3.4 times the memory of inference.
- Training with our own backward pass: 97.27% on the test set after 5 epochs, exactly as in the reel and the demo;
  67 rows of $W_1$ (the pixels that are always blank) never change.

## Review questions
1. Why must the forward pass keep $h_1$ and $h_2$ during training, but not for inference alone?
2. In the demo, $x$ goes into both $z_1$ and $z_2$. Why is $\partial L/\partial x$ the sum of the two branches?
3. Does the gradient of $W_3$ need the ReLU mask? Which result of section 7 does that explain?
4. Why can a gradient check in float32, or with $h = 0.1$, report an error even though the backward pass is correct?
5. Why does this network's backward pass cost only about 1.08 times the forward pass, rather than twice as much?
6. After 5 epochs, 67 rows of $W_1$ are still exactly equal to their initial values. Why?

<details><summary>Hints and answers</summary>

1. $\partial W_2 = h_1^\top \partial a_2$ and the ReLU mask $[h > 0]$ both need them. At inference, once a layer is
   computed, its input can be discarded.
2. Nudging $x$ changes both branches, and the effects along each path to $L$ add up (the multivariable chain rule).
3. No: $\partial W_3 = h_2^\top \partial z$ is computed before going through any ReLU gate. That is why forgetting the
   ReLU gate still lets $W_3$ and $b_3$ pass the check; only the tensors before the gate are wrong.
4. With a large $h$, the interval $[\theta - h, \theta + h]$ can contain a ReLU kink. float32 has only about 7
   significant digits, so the subtraction $\mathcal L(\theta + h) - \mathcal L(\theta - h)$ loses almost all of its
   precision.
5. Each layer needs $\partial W$ and $\partial h$, and each costs as much as that layer's forward pass. But layer 1
   (100,352 of the 109,184 MACs) doesn't need $\partial x$, because nobody learns the pixels.
6. Row $i$ of $\partial W_1 = x^\top \partial a_1$ is always multiplied by $x_i$; those 67 pixels are blank in all
   60,000 training images, so their gradients are always exactly 0.
</details>

## Further reading
- MLSysBook, [*Neural Computation*](https://mlsysbook.ai/vol1/nn_computation/nn_computation.html): the backward-pass
  formulas (equations 22–24), the worked example by hand and the training-vs-inference memory ledger (Table 8) used in
  this lesson.
- D. Rumelhart, G. Hinton & R. Williams (1986), [Learning representations by back-propagating errors](https://doi.org/10.1038/323533a0).
- A. Griewank (2012), [Who Invented the Reverse Mode of Differentiation?](https://doi.org/10.4171/dms/6/38): Linnainmaa 1970.
- P. Werbos (1982), [Applications of advances in nonlinear sensitivity analysis](https://doi.org/10.1007/BFb0006203).
- CS231n, [Backpropagation, Intuitions](https://cs231n.github.io/optimization-2/) (patterns in backward flow,
  dimension analysis) and [Gradient checks](https://cs231n.github.io/neural-networks-3/) (relative error, float64,
  kinks).
- A. Karpathy, [micrograd](https://github.com/karpathy/micrograd) and the lecture
  [Becoming a Backprop Ninja](https://youtu.be/q8SA3rM6ckI): writing the backward pass of a whole MLP by hand.
- C. Olah (2015), [Calculus on Computational Graphs: Backpropagation](https://colah.github.io/posts/2015-08-Backprop/).
- T. Chen et al. (2016), [Training Deep Nets with Sublinear Memory Cost](https://arxiv.org/abs/1604.06174):
  activation checkpointing.

**Next lesson:** [Lesson 07 · Momentum, Adam & AdamW](https://d3qr1cjln6qqtk.cloudfront.net/en/lessons/bo-toi-uu/): keep
the backward pass as it is, and improve the update step.

In [ ]:
import platform

import matplotlib

print(f"Python {platform.python_version()} · numpy {np.__version__} · matplotlib {matplotlib.__version__}")